# NB15 — Paper outputs

        **CPU only. ~10 minutes.**


> **New here?** Read `05_PLAIN_ENGLISH_GUIDE.md` first — it explains what this
> project is measuring and why, without jargon. This notebook assumes you have.


        Assembles every table and figure, totals the energy, and — the part that
        matters for credibility — builds a manifest mapping **every number to the
        run that produced it**.

        The engineering spec's first reproducibility requirement is that every
        number in the paper maps to a run ID. This notebook makes that checkable
        rather than aspirational.

In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# This writes two Python files into the session and imports them. Nothing here
# touches the GPU or the network beyond installing three small packages.
#
#   msc_lib   9947cabf3e22   the pipeline: HuggingFace sync, model zoo,
#                              measurement, training, the method
#   msc_core  2571e11b4f7e   the reference maths: the MSC definition and
#                              every statistic in the paper
#
# Both are generated from KD/src by build_notebooks.py. Editing them HERE does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle images already ship torch, pandas and sklearn. These three vary by
# image version, so we check rather than assume.
#   pyarrow  writes the per-image measurement tables (Parquet)
#   pynvml   reads GPU power/temperature/utilisation directly
#   fvcore   counts FLOPs, which is how compute cost is defined
for _pkg in ('pyarrow', 'pynvml', 'fvcore', 'psutil'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg,
                        '--break-system-packages'], check=False)

_LIB = (
    '...',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]

import msc_lib as msc
import msc_core

print(f'[BOOT] msc_lib v{msc.__version__} ready  (torch available: {msc._TORCH_OK})')
print(f'[BOOT] artifact space: {msc.WORK_ROOT}   scratch space: {msc.SCRATCH_ROOT}')

## Step 1 — Session

In [ ]:
ACCOUNT = 'acct1'      # <<< CHANGE ME
sess = msc.Session(account=ACCOUNT, phase='paper', dataset='cifar100', enable_hf=True)
sess.sync_state(include_checkpoints=False, verbose=True)
import pandas as pd, numpy as np, matplotlib.pyplot as plt

## Step 2 — Everything that ran

In [ ]:
ledger = sess.status()
display(ledger)
if len(ledger):
    msc.save_analysis(sess.data_dir, 'run_ledger', ledger, sess.hub)
    print(f"\n{len(ledger)} runs   "
          f"{int((ledger.state == 'completed').sum())} completed")

## Step 3 — Table 1: the atlas

Accuracy against published references. The `gap` column is the audit —
anything above 1.0 means that model is under-trained and every
measurement from it is suspect.

In [ ]:
rows = []
for d in sorted(sess.runs_dir.iterdir()) if sess.runs_dir.exists() else []:
    s = msc.read_json(d / 'summary.json', default=None)
    if not s or s.get('status') != 'completed':
        continue
    rows.append({k: s.get(k) for k in
                 ('run_id', 'arch', 'family', 'seed', 'best_accuracy',
                  'final_accuracy_top5', 'reference_accuracy',
                  'accuracy_gap_vs_reference', 'num_parameters', 'full_flops',
                  'total_time_sec', 'total_energy_kwh', 'total_co2_kg',
                  'num_epochs_run', 'config_hash')})
t1 = pd.DataFrame(rows)
if not len(t1):
    print('No completed runs found. Nothing to tabulate yet.')
if len(t1):
    t1['params_M'] = (t1.num_parameters / 1e6).round(2)
    t1['GFLOPs'] = (t1.full_flops / 1e9).round(3)
    t1['acc_pct'] = (t1.best_accuracy * 100).round(2)
    t1 = t1.sort_values(['family', 'arch', 'seed'])
    display(t1[['run_id', 'arch', 'seed', 'acc_pct', 'reference_accuracy',
                'accuracy_gap_vs_reference', 'params_M', 'GFLOPs',
                'total_energy_kwh']])
    msc.save_analysis(sess.data_dir, 'table1_atlas', t1, sess.hub)

    agg = (t1.groupby('arch').agg(acc_mean=('acc_pct', 'mean'),
                                  acc_std=('acc_pct', 'std'),
                                  n_seeds=('seed', 'count'),
                                  params_M=('params_M', 'first'),
                                  GFLOPs=('GFLOPs', 'first')).round(3))
    print('\nmean +/- std across seeds (single-seed numbers do not go in a paper):')
    display(agg)
    msc.save_analysis(sess.data_dir, 'table1_atlas_aggregated',
                      agg.reset_index(), sess.hub)

## Step 4 — Training telemetry summary

Everything we recorded per epoch, summarised. This is what lets you
answer "why was that architecture slow?" months later.

In [ ]:
rows = []
for d in sorted(sess.runs_dir.iterdir()) if sess.runs_dir.exists() else []:
    h = d / 'metrics' / 'epochs.csv'
    if not h.exists():
        continue
    try:
        df = pd.read_csv(h)
    except Exception:
        continue
    if df.empty:
        continue
    rec = {'run_id': d.name, 'epochs': len(df)}
    for c, agg in (('epoch_time_sec', 'median'), ('throughput_img_s', 'median'),
                   ('dataload_frac', 'median'), ('gpu_util_mean_pct', 'mean'),
                   ('gpu_temp_max_c', 'max'), ('peak_vram_mb', 'max'),
                   ('grad_norm_mean', 'median'), ('update_to_weight_ratio', 'median'),
                   ('step_time_p99_ms', 'median'), ('nan_or_inf_batches', 'sum'),
                   ('cumulative_energy_kwh', 'max')):
        if c in df.columns:
            rec[c] = float(getattr(df[c], agg)())
    rows.append(rec)
tel = pd.DataFrame(rows)
if len(tel):
    display(tel.round(3))
    msc.save_analysis(sess.data_dir, 'table_training_telemetry', tel, sess.hub)
    if 'dataload_frac' in tel:
        starved = tel[tel.dataload_frac > 0.3]
        if len(starved):
            print('\nThese runs spent >30% of their time waiting for data --')
            print('the GPU was idle. Worth knowing before scaling up:')
            display(starved[['run_id', 'dataload_frac', 'gpu_util_mean_pct']])
    if 'nan_or_inf_batches' in tel:
        bad = tel[tel.nan_or_inf_batches > 0]
        if len(bad):
            print('\nRuns with NaN/Inf batches (silent AMP failures):')
            display(bad[['run_id', 'nan_or_inf_batches']])

## Step 5 — Energy and carbon

Reported as measurement methodology, never claimed as a contribution.
FLOPs is our primary efficiency metric; FLOP-based proxies underestimate
real energy by 2–6× because of memory traffic and kernel-launch overhead,
which is exactly why we sample power directly.

In [ ]:
if len(t1):
    e = t1.groupby('family').agg(
        total_kwh=('total_energy_kwh', 'sum'),
        total_co2_kg=('total_co2_kg', 'sum'),
        total_gpu_h=('total_time_sec', lambda s: s.sum() / 3600.0),
        runs=('run_id', 'count')).round(4)
    display(e)
    print(f"\nPROJECT TOTAL: {t1.total_energy_kwh.sum():.3f} kWh | "
          f"{t1.total_co2_kg.sum():.3f} kg CO2 | "
          f"{t1.total_time_sec.sum()/3600:.1f} T4-hours")
    msc.save_analysis(sess.data_dir, 'table_energy_accounting',
                      e.reset_index(), sess.hub)

## Step 5b — Combined tables across every run

Concatenates every run's `metrics/epochs.csv` and `metrics/final.csv`
into two repo-level files. This is the "all combined" view — one CSV you
can open and see the entire project in, without walking folders.

- `tables/all_epochs.csv` — every epoch of every run
- `tables/all_final.csv` — one row per run
- `tables/atlas_summary.csv` — mean ± std across seeds

In [ ]:
ep_frames, fi_frames = [], []
for d in sorted(sess.runs_dir.iterdir()) if sess.runs_dir.exists() else []:
    e, f = d / 'metrics' / 'epochs.csv', d / 'metrics' / 'final.csv'
    if e.exists():
        try:
            df = pd.read_csv(e)
            df['run_id'] = df.get('run_id', d.name)
            ep_frames.append(df)
        except Exception as ex:
            print(f'  {d.name} epochs: {ex}')
    if f.exists():
        try:
            fi_frames.append(pd.read_csv(f))
        except Exception as ex:
            print(f'  {d.name} final: {ex}')

tdir = msc.ensure_dir(sess.data_dir / 'tables')
if ep_frames:
    all_ep = pd.concat(ep_frames, ignore_index=True, sort=False)
    all_ep.to_csv(tdir / 'all_epochs.csv', index=False)
    print(f'tables/all_epochs.csv : {len(all_ep):,} rows x {len(all_ep.columns)} cols '
          f'from {len(ep_frames)} runs')
    display(all_ep.head(3))
if fi_frames:
    all_fi = pd.concat(fi_frames, ignore_index=True, sort=False)
    all_fi.to_csv(tdir / 'all_final.csv', index=False)
    print(f'tables/all_final.csv  : {len(all_fi)} runs x {len(all_fi.columns)} cols')
    display(all_fi)

    num = all_fi.select_dtypes('number').columns
    summ = (all_fi.groupby('arch')[list(num)].agg(['mean', 'std', 'count'])
            if ('arch' in all_fi.columns and len(all_fi)) else pd.DataFrame())
    if len(summ):
        summ.to_csv(tdir / 'atlas_summary.csv')
        print(f'tables/atlas_summary.csv : mean +/- std across seeds')

if not ep_frames and not fi_frames:
    print('Nothing to combine yet -- no run has written metrics/ on this account.')
    print('Run sess.sync_state() first, or wait for training to produce epochs.')

if sess.hub.enabled and (ep_frames or fi_frames):
    sess.hub.hub.enqueue_dir(tdir, 'tables')
    sess.hub.flush(timeout=600)

## Step 6 — Every computed statistic

In [ ]:
adir = sess.data_dir / 'analysis'
found = sorted(p.name for p in adir.glob('*.csv')) if adir.exists() else []
print(f'{len(found)} analysis tables saved:')
for f in found:
    print('  ', f)
figs = sorted(p.name for p in (sess.data_dir / 'paper' / 'figures').glob('*.png')) \
    if (sess.data_dir / 'paper' / 'figures').exists() else []
print(f'\n{len(figs)} figures:')
for f in figs:
    print('  ', f)

## Step 7 — Provenance manifest

Every artifact, its size, its checksum, and the run that made it.

In [ ]:
prov = msc.provenance_manifest(sess.data_dir, sess.hub)
print(f'{len(prov)} artifacts tracked')
display(prov.head(30))

## Step 8 — Model card

In [ ]:
lines = [
    '# MSC — Minimum Sufficient Compute', '',
    'Artifacts for *Is Compute Difficulty Architecture-Agnostic? Measuring and',
    'Distilling Per-Sample Minimum Sufficient Computation*.', '',
    f'Generated {msc.now_iso()} by msc_lib v{msc.__version__}.', '',
    '## Repositories', '',
    f'- `{msc.HF_REPO}` — everything, one folder per run', '',
    '## What MSC is', '',
    'The smallest cost-normalised configuration at which a network\'s decision has',
    '*stably settled* to its full-compute decision, defined uniformly over depth,',
    'resolution and precision reduction. Stability means the decision agrees at that',
    'budget **and every larger one** — predictions under compute reduction are not',
    'monotone, so a naive minimum records an accident rather than a property.', '',
    '## Compute grid', '',
    f'- depth: exits at {list(msc.DEPTH_FRACTIONS)} of network depth',
    f'- resolution: {list(msc.RESOLUTIONS)} px, measured natively AND via a',
    '  downsample-upsample proxy (the proxy cost model is labelled idealised)',
    f'- precision: {list(msc.PRECISIONS)}, simulated by fake quantisation;',
    '  cost priced analytically as bits/32, never as measured latency',
    f'- confidence thresholds: tau in {list(msc.TAU_GRID)} — all results are tau-curves',
    '', '## Per-image table schema', '', '```',
    'sample_idx, label,',
    'pred_d1..d5    top1p_d1..d5    top2p_d1..d5     depth',
    'pred_rn1..rn5  top1p_rn1..rn5  top2p_rn1..rn5   resolution (native)',
    'pred_rp1..rp5  top1p_rp1..rp5  top2p_rp1..rp5   resolution (proxy)',
    'pred_q1..q5    top1p_q1..q5    top2p_q1..q5     precision',
    'msp, margin, entropy, ce_loss, el2n, forget_events, pred_depth',
    'sample_order_hash, run_id, split', '```', '',
    'Every table carries `sample_order_hash`. Tables whose hashes differ are not',
    'row-aligned and must not be correlated.', '',
    '## Telemetry recorded per epoch', '',
    'Losses and accuracies; learning rate per group; gradient norm mean/max/p95;',
    'gradient-clip hit rate; weight norm; update-to-weight ratio; AMP scale;',
    'NaN/Inf batch count; epoch/train/eval time; dataload vs compute split;',
    'step-time p50/p90/p99; throughput; VRAM allocated/reserved/peak; GPU',
    'utilisation and temperature; CPU and RAM; free disk; energy in J/kWh and CO2',
    'per epoch and cumulative. Plus raw power samples at 10 Hz, system samples at',
    '1 Hz, and a downsampled per-step trace.', '',
    '## Reproducibility', '',
    '- `config.yaml` frozen at run start, sha256-hashed, asserted on resume',
    '- checkpoints carry optimizer, scheduler, AMP scaler and all four RNG streams',
    '- 3 seeds per headline number, mean +/- std',
    '- every artifact mapped to a run_id in `paper/provenance.csv`',
    '- work split across accounts by a deterministic cost-balanced scheduler;',
    '  each run records which worker produced it', '',
]
if len(t1):
    lines += ['## Atlas results', '',
              t1[['arch', 'seed', 'acc_pct', 'reference_accuracy',
                  'params_M', 'GFLOPs']].to_markdown(index=False), '']
lines += ['## Limitations', '',
          '- Per-image routing gives **no wall-clock speedup under batched',
          '  inference** unless the batch is split by route. The deployment claim',
          '  is scoped to batch-1 / edge / streaming.',
          '- INT4 and INT6 are simulated; no T4 kernel exists to time them.',
          '- The resolution proxy runs at 32 px; its cost is an idealised model.',
          '- Risk control is calibrated at epsilon=0.03 on a 5,000-image holdout,',
          '  because epsilon=0.01 would need ~14,979 calibration images and the',
          '  CIFAR-100 test set has 10,000.',
          '- Energy is measurement methodology, not a contribution.',
          '- T4-only hardware; CIFAR-100 scale.', '']

card = sess.data_dir / 'README.md'
msc.atomic_write_text(card, '\n'.join(lines))
if sess.hub.enabled:
    sess.hub.hub.enqueue(card, 'README.md')
print('\n'.join(lines[:45]))

## Step 9 — Finish

In [ ]:
# === Push everything and stop ==============================================
# Blocks until HuggingFace confirms. Safe to re-run.
sess.finish()